# `combine_QG` — build per-protein `QG.npy` and 1D free-energy profiles (minor5)For **every protein in all four groups** (`NSC_ENT`, `SC_NENT`, `SC_ENT`, `NSC_NENT`) this notebook1. creates an `analysis/` folder next to `opt/`, `quench/`, `setup/`;2. combines the *existing* per-replica order parameters   `quench/traj/OP/Q/traj_<i>.Q` and `quench/traj/OP/G/traj_<i>.G`   into a single array `analysis/QG.npy` of shape **`(n, 19999, 2)`** —   *n* = number of non-mirror replicas, 19999 frames, `[:, :, 0] = Q`, `[:, :, 1] = G`;3. computes a **1D free energy along Q**, `F(Q) = -ln P(Q)` in units of *kT*,   saved as `analysis/Q_free_energy.csv` + `.png` (and the same for G).**Conventions carried over from the earlier protein groups**(`cg_sim_300K/sc_ent/0_P05744/analysis/combine_GQ.py`):* the raw files hold 20 000 frames (`Frame` 0–19999); the **last frame is dropped**  so the stored array is 19999 frames long, matching the previous data sets;* only the *last* column of each file is kept (`Q` and `G` respectively);* mirror-image replicas were already excluded upstream, so their `.Q`/`.G` files  simply do not exist — the replica list is therefore **discovered from disk**  rather than assumed to be `range(50)`.**Data safety** — nothing outside the new `analysis/` folders is ever written.Existing `QG.npy` files are skipped unless `FORCE = True`.

In [1]:

import os
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

## 1. Configuration

In [2]:

# --- paths -----------------------------------------------------------------
SIM_ROOT = Path("/storage/group/epo2/default/qzv5006/Research_data/Aging_project"
                "/R1_analysis/minor5/sim")
GROUPS = ["NSC_ENT", "SC_NENT", "SC_ENT", "NSC_NENT"]

# --- array layout (must match the earlier protein groups) ------------------
N_FRAMES = 19999        # frames kept per replica (raw files have 20000 -> drop last)
N_OP = 2                # order parameters stored: [Q, G]
IDX_Q, IDX_G = 0, 1

# --- free-energy settings --------------------------------------------------
N_BINS = 75             # same binning as combine_GQ.py
Q_RANGE = (0.0, 1.0)    # fixed grid so F(Q) is directly comparable across proteins
G_RANGE = None          # None -> data-driven range (G spans a narrow, system-specific window)

# --- run control -----------------------------------------------------------
FORCE = False           # True -> rebuild QG.npy even if it already exists
N_WORKERS = 8           # threads used for the per-replica file reads
SHOW_PLOTS = False      # False -> figures are written to disk but not displayed inline

if not SHOW_PLOTS:
    matplotlib.use("Agg")

print("sim root :", SIM_ROOT)
print("exists   :", SIM_ROOT.is_dir())

sim root : /storage/group/epo2/default/qzv5006/Research_data/Aging_project/R1_analysis/minor5/sim
exists   : True


## 2. Discovery helpers`quench/traj/OP/Q/` and `quench/traj/OP/G/` are the source of truth for whichreplicas survived the mirror screen. A replica is used only when **both** its`.Q` and `.G` file are present; any mismatch is reported rather than silentlydropped.

In [3]:

def list_proteins(group):
    """Protein directories of `group`, sorted by their numeric prefix.

    Skips two kinds of directory, matching every shell script in scripts/:

    * ``REMOVED_<NN>_<ACC>`` -- proteins dropped as transmembrane or secretory
      and replaced. Their finished runs are parked in place, so they still hold
      a full trajectory and OP set and were silently being analysed alongside
      the replacements that superseded them.
    * directories with no ``setup/`` -- analysis-only imports from cg_sim_300K,
      copied for their results rather than to be re-run.
    """
    gdir = SIM_ROOT / group
    return sorted((p for p in gdir.iterdir()
                   if p.is_dir()
                   and not p.name.startswith("REMOVED_")
                   and (p / "setup").is_dir()),
                  key=lambda p: p.name)


N_COPIES = 50           # replicas per protein before the mirror screen


def kept_replicas(protein_dir):
    """Replicas that SHOULD exist: N_COPIES minus the mirror-screened ones.

    12_gq_runlist.sh never generates OP files for a replica topo flagged as a
    mirror, so the target is not always N_COPIES.
    """
    v = protein_dir / "quench" / "traj" / "mirror" / "verdicts.tsv"
    n = N_COPIES
    if v.is_file():
        for ln in v.read_text(errors="replace").splitlines()[1:]:
            f = ln.split("\t")
            if len(f) > 5 and f[5].strip() == "True":
                n -= 1
    return n


def op_dirs(protein_dir):
    """(Q dir, G dir) holding the per-replica order-parameter files."""
    op = protein_dir / "quench" / "traj" / "OP"
    return op / "Q", op / "G"


def replica_indices(protein_dir):
    """
    Replica indices that have BOTH a .Q and a .G file, sorted numerically.

    Returns (common, q_only, g_only) so that any asymmetry is visible.
    """
    qdir, gdir = op_dirs(protein_dir)
    q_idx = {int(p.stem.split("_")[-1]) for p in qdir.glob("traj_*.Q")} if qdir.is_dir() else set()
    g_idx = {int(p.stem.split("_")[-1]) for p in gdir.glob("traj_*.G")} if gdir.is_dir() else set()
    return sorted(q_idx & g_idx), sorted(q_idx - g_idx), sorted(g_idx - q_idx)

## 3. Loading and stacking`pandas.read_csv` with `usecols` is used instead of `np.loadtxt` — it parses thesame comma-separated files roughly an order of magnitude faster, which mattersat ~4 800 files. The kept column is selected **by name** (`Q` / `G`), which isthe last column in both formats.

In [4]:

def _read_op(path, column):
    """Last-column order parameter of one .Q/.G file, truncated to N_FRAMES.

    Raises ValueError if the file is shorter than N_FRAMES, so a truncated or
    still-running replica can never be padded with garbage.
    """
    s = pd.read_csv(path, usecols=[column])[column].to_numpy(dtype=np.float64)
    if s.shape[0] < N_FRAMES:
        raise ValueError(f"{path} has {s.shape[0]} frames, expected >= {N_FRAMES}")
    return s[:N_FRAMES]          # drop the trailing frame(s)


def build_qg(protein_dir, workers=N_WORKERS):
    """
    Stack every non-mirror replica of one protein into an (n, N_FRAMES, 2) array.

    Returns (QG, info) where `info` records what was found and any anomalies.
    """
    qdir, gdir = op_dirs(protein_dir)
    common, q_only, g_only = replica_indices(protein_dir)

    info = {
        "protein": protein_dir.name,
        "n_replicas": len(common),
        "replicas": common,
        "q_without_g": q_only,
        "g_without_q": g_only,
        "raw_frames": None,
        "errors": [],
    }
    if not common:
        info["errors"].append("no paired .Q/.G files found")
        return None, info

    # frame count of the raw files, for the record (before truncation)
    info["raw_frames"] = int(
        pd.read_csv(qdir / f"traj_{common[0]}.Q", usecols=["Q"]).shape[0]
    )

    def load(i):
        return _read_op(qdir / f"traj_{i}.Q", "Q"), _read_op(gdir / f"traj_{i}.G", "G")

    QG = np.empty((len(common), N_FRAMES, N_OP), dtype=np.float64)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for k, (q, g) in enumerate(ex.map(load, common)):
            QG[k, :, IDX_Q] = q
            QG[k, :, IDX_G] = g

    if not np.isfinite(QG).all():
        info["errors"].append(f"{int((~np.isfinite(QG)).sum())} non-finite values")
    return QG, info

## 4. Build `analysis/QG.npy` for every proteinEach protein gets```<group>/<NN_UNIPROT>/analysis/QG.npy        (n, 19999, 2)  float64<group>/<NN_UNIPROT>/analysis/QG_info.json  provenance: replica indices, shape, source```The JSON sidecar records *which* replica indices went into which row of thearray — without it the row order of a variable-length replica set isirrecoverable.

In [5]:

records = []

for group in GROUPS:
    for protein_dir in list_proteins(group):
        adir = protein_dir / "analysis"
        adir.mkdir(exist_ok=True)                 # never touches opt/ quench/ setup/
        out = adir / "QG.npy"

        if out.exists() and not FORCE:
            QG = np.load(out, mmap_mode="r")
            records.append(dict(group=group, protein=protein_dir.name,
                                n_replicas=QG.shape[0], frames=QG.shape[1],
                                status="exists (skipped)", note=""))
            print(f"{group}/{protein_dir.name:<12} skip   {QG.shape}")
            continue

        # job_gq.sh writes .Q before .G and advances all blocks in parallel,
        # so a protein mid-run has only a PARTIAL paired set. Building now
        # would write a truncated QG.npy -- and because FORCE defaults to
        # False, that truncated array would then be skipped, never rebuilt, on
        # every later run. Refuse and say so, so the notebook is safe to run
        # while GQ is still in flight and simply picks the protein up later.
        _common, _q_only, _g_only = replica_indices(protein_dir)
        _expected = kept_replicas(protein_dir)
        if _common and len(_common) < _expected:
            records.append(dict(group=group, protein=protein_dir.name,
                                n_replicas=len(_common), frames=0,
                                status="incomplete (skipped)",
                                note=f"{len(_common)}/{_expected} paired replicas; "
                                     f"job_gq.sh still running -- re-run later"))
            print(f"{group}/{protein_dir.name:<12} wait   "
                  f"{len(_common)}/{_expected} paired replicas")
            continue

        QG, info = build_qg(protein_dir)
        if QG is None:
            records.append(dict(group=group, protein=protein_dir.name, n_replicas=0,
                                frames=0, status="FAILED", note="; ".join(info["errors"])))
            print(f"{group}/{protein_dir.name:<12} FAIL   {info['errors']}")
            continue

        np.save(out, QG)
        info.update(shape=list(QG.shape),
                    source=str(protein_dir / "quench" / "traj" / "OP"),
                    frames_kept=N_FRAMES,
                    note=f"row k of QG.npy corresponds to replicas[k]; "
                         f"axis 2 = [Q, G]; last raw frame dropped")
        (adir / "QG_info.json").write_text(json.dumps(info, indent=2))

        note = "; ".join(info["errors"])
        if info["q_without_g"] or info["g_without_q"]:
            note = (note + "; " if note else "") + \
                   f"unpaired Q={info['q_without_g']} G={info['g_without_q']}"
        records.append(dict(group=group, protein=protein_dir.name,
                            n_replicas=QG.shape[0], frames=QG.shape[1],
                            status="written", note=note))
        print(f"{group}/{protein_dir.name:<12} ok     {QG.shape}"
              f"{'  <-- ' + note if note else ''}")

manifest = pd.DataFrame(records)
manifest

NSC_ENT/01_P38910    skip   (35, 19999, 2)
NSC_ENT/02_P38828    skip   (50, 19999, 2)
NSC_ENT/03_P38146    skip   (50, 19999, 2)
NSC_ENT/04_P32835    skip   (50, 19999, 2)
NSC_ENT/05_P38858    skip   (50, 19999, 2)
NSC_ENT/06_P03962    skip   (50, 19999, 2)
NSC_ENT/07_P38816    skip   (50, 19999, 2)
NSC_ENT/08_P32179    skip   (50, 19999, 2)
NSC_ENT/09_P28834    skip   (50, 19999, 2)
NSC_ENT/10_Q12283    skip   (50, 19999, 2)
NSC_ENT/11_P40530    skip   (50, 19999, 2)
NSC_ENT/12_Q03305    skip   (50, 19999, 2)
NSC_ENT/13_P38755    skip   (50, 19999, 2)
NSC_ENT/14_P52917    skip   (50, 19999, 2)
NSC_ENT/15_P24521    skip   (50, 19999, 2)
NSC_ENT/16_P53960    skip   (50, 19999, 2)
NSC_ENT/17_P53255    skip   (50, 19999, 2)
NSC_ENT/18_P08536    skip   (50, 19999, 2)
NSC_ENT/19_P10869    skip   (50, 19999, 2)
NSC_ENT/20_P37012    skip   (50, 19999, 2)
NSC_ENT/21_P53235    skip   (50, 19999, 2)
NSC_ENT/22_P34243    skip   (50, 19999, 2)
NSC_ENT/23_P23337    skip   (50, 19999, 2)


NSC_ENT/24_P25039    ok     (50, 19999, 2)
SC_NENT/01_Q6Q560    skip   (50, 19999, 2)
SC_NENT/02_P48164    skip   (50, 19999, 2)
SC_NENT/03_P36015    skip   (50, 19999, 2)
SC_NENT/04_P49775    skip   (50, 19999, 2)
SC_NENT/05_Q12522    skip   (50, 19999, 2)
SC_NENT/06_P38886    skip   (50, 19999, 2)
SC_NENT/07_P29311    skip   (50, 19999, 2)
SC_NENT/08_P32496    skip   (50, 19999, 2)
SC_NENT/09_P53981    skip   (50, 19999, 2)
SC_NENT/10_P26321    skip   (50, 19999, 2)
SC_NENT/11_Q04491    skip   (50, 19999, 2)
SC_NENT/12_P24280    skip   (50, 19999, 2)
SC_NENT/13_P00358    skip   (50, 19999, 2)
SC_NENT/14_P38431    skip   (50, 19999, 2)
SC_NENT/15_P53691    skip   (50, 19999, 2)
SC_NENT/16_P38998    skip   (50, 19999, 2)
SC_NENT/17_P41338    skip   (50, 19999, 2)
SC_NENT/18_Q08421    skip   (50, 19999, 2)
SC_NENT/19_P32527    skip   (50, 19999, 2)
SC_NENT/20_P41807    skip   (50, 19999, 2)
SC_NENT/21_P39976    skip   (50, 19999, 2)
SC_NENT/22_P14904    skip   (50, 19999, 2)
SC_NENT/23_

,group,protein,n_replicas,frames,status,note
0,NSC_ENT,01_P38910,35,19999,exists (skipped),
1,NSC_ENT,02_P38828,50,19999,exists (skipped),
2,NSC_ENT,03_P38146,50,19999,exists (skipped),
3,NSC_ENT,04_P32835,50,19999,exists (skipped),
4,NSC_ENT,05_P38858,50,19999,exists (skipped),
5,NSC_ENT,06_P03962,50,19999,exists (skipped),
6,NSC_ENT,07_P38816,50,19999,exists (skipped),
7,NSC_ENT,08_P32179,50,19999,exists (skipped),
8,NSC_ENT,09_P28834,50,19999,exists (skipped),
9,NSC_ENT,10_Q12283,50,19999,exists (skipped),


In [6]:

# --- sanity check: shape convention and replica-count outliers -------------
bad_shape = manifest[(manifest.frames != N_FRAMES) & (manifest.status != "FAILED")]
print("proteins not at", N_FRAMES, "frames:", len(bad_shape))
if len(bad_shape):
    display(bad_shape)

print("\nreplica-count distribution:")
print(manifest.n_replicas.value_counts().sort_index().to_string())
print("\nproteins with fewer than the modal replica count "
      "(mirror-image trajectories removed upstream):")
modal = manifest.n_replicas.mode().iat[0]
display(manifest[manifest.n_replicas < modal][["group", "protein", "n_replicas"]])

proteins not at 19999 frames: 0

replica-count distribution:
n_replicas
35     1
50    55

proteins with fewer than the modal replica count (mirror-image trajectories removed upstream):


,group,protein,n_replicas
0,NSC_ENT,01_P38910,35


## 5. 1D free energy along Q$F(x) = -\ln P(x)$ in units of *kT*, with the minimum shifted to zero. Emptybins become `NaN` rather than `inf` so they simply leave a gap in the curve.Q is histogrammed on the **fixed grid `Q_RANGE`** so the profiles of differentproteins share an x-axis (the earlier single-protein script used the data range;with 48 proteins a common grid is what makes them comparable). G keeps adata-driven range because its scale is system-specific.

In [7]:

def free_energy_1d(x, bins=N_BINS, rng=None):
    """Return (bin_centers, F/kT, P) for a 1D reaction coordinate."""
    x = np.asarray(x).ravel()
    x = x[np.isfinite(x)]
    hist, edges = np.histogram(x, bins=bins, range=rng, density=True)
    p = np.where(hist == 0, np.nan, hist)       # avoid log(0)
    F = -np.log(p)
    F -= np.nanmin(F)                            # shift global minimum to 0 kT
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, F, hist


def plot_free_energy(centers, F, label, xlabel, title, outfile):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(centers, F, lw=2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Free Energy (kT)")
    ax.set_title(title)
    ax.grid(True, alpha=0.4)
    fig.tight_layout()
    fig.savefig(outfile, dpi=300)
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)

In [8]:

fe_records = []

for group in GROUPS:
    for protein_dir in list_proteins(group):
        adir = protein_dir / "analysis"
        npy = adir / "QG.npy"
        if not npy.exists():
            continue

        QG = np.load(npy)                      # (n, 19999, 2)
        pooled = QG.reshape(-1, N_OP)          # pool every frame of every replica

        # ---- Q ------------------------------------------------------------
        cQ, FQ, _ = free_energy_1d(pooled[:, IDX_Q], rng=Q_RANGE)
        pd.DataFrame({"Q": cQ, "F_kT": FQ}).to_csv(adir / "Q_free_energy.csv", index=False)
        plot_free_energy(cQ, FQ, "Q", "Reaction Coordinate (Q)",
                         f"{group}/{protein_dir.name}  (n={QG.shape[0]})",
                         adir / "Q_free_energy.png")

        # ---- G ------------------------------------------------------------
        cG, FG, _ = free_energy_1d(pooled[:, IDX_G], rng=G_RANGE)
        pd.DataFrame({"G": cG, "F_kT": FG}).to_csv(adir / "G_free_energy.csv", index=False)
        plot_free_energy(cG, FG, "G", "Reaction Coordinate (G)",
                         f"{group}/{protein_dir.name}  (n={QG.shape[0]})",
                         adir / "G_free_energy.png")

        # ---- cheap folding diagnostics ------------------------------------
        tail = QG[:, -2000:, IDX_Q]            # last ~100 ns of each replica
        fe_records.append(dict(
            group=group, protein=protein_dir.name, n_replicas=QG.shape[0],
            Q_mean=float(pooled[:, IDX_Q].mean()),
            Q_tail_mean=float(tail.mean()),
            Q_at_Fmin=float(cQ[np.nanargmin(FQ)]),
            frac_traj_tailQ_gt_0p6=float((tail.mean(axis=1) > 0.6).mean()),
            G_max=float(pooled[:, IDX_G].max()),
        ))
        print(f"{group}/{protein_dir.name:<12} F(Q) written  "
              f"Q_min at {cQ[np.nanargmin(FQ)]:.3f}")

fe = pd.DataFrame(fe_records)
fe

NSC_ENT/01_P38910    F(Q) written  Q_min at 0.940


NSC_ENT/02_P38828    F(Q) written  Q_min at 0.807


NSC_ENT/03_P38146    F(Q) written  Q_min at 0.900


NSC_ENT/04_P32835    F(Q) written  Q_min at 0.953


NSC_ENT/05_P38858    F(Q) written  Q_min at 0.953


...

NSC_NENT/15_P27796    F(Q) written  Q_min at 0.793


NSC_NENT/16_P16387    F(Q) written  Q_min at 0.967


NSC_NENT/17_P50079    F(Q) written  Q_min at 0.940


NSC_NENT/24_Q12387    F(Q) written  Q_min at 0.967


,group,protein,n_replicas,Q_mean,Q_tail_mean,Q_at_Fmin,frac_traj_tailQ_gt_0p6,G_max
0,NSC_ENT,01_P38910,35,0.896107,0.909025,0.940000,1.00,0.07853
1,NSC_ENT,02_P38828,50,0.812044,0.821468,0.806667,1.00,0.11009
2,NSC_ENT,03_P38146,50,0.858756,0.901779,0.900000,1.00,0.27487
3,NSC_ENT,04_P32835,50,0.848285,0.935700,0.953333,1.00,0.26360
4,NSC_ENT,05_P38858,50,0.807711,0.954413,0.953333,1.00,0.19904
5,NSC_ENT,06_P03962,50,0.913866,0.948938,0.966667,1.00,0.09926
6,NSC_ENT,07_P38816,50,0.852297,0.902342,0.913333,1.00,0.13974
7,NSC_ENT,08_P32179,50,0.861748,0.900880,0.980000,1.00,0.13900
8,NSC_ENT,09_P28834,50,0.863338,0.911951,0.953333,1.00,0.21123
9,NSC_ENT,10_Q12283,50,0.870860,0.936888,0.953333,1.00,0.13254


## 6. Group-level overviewAll per-protein `F(Q)` curves on one axis per group — a quick read on whichproteins actually reached a folded basin and which are stuck at low Q.

In [9]:

fig, axes = plt.subplots(1, len(GROUPS), figsize=(6.5 * len(GROUPS), 5),
                         sharex=True, sharey=True)
axes = np.atleast_1d(axes)

for ax, group in zip(axes, GROUPS):
    for protein_dir in list_proteins(group):
        csv = protein_dir / "analysis" / "Q_free_energy.csv"
        if not csv.exists():
            continue
        df = pd.read_csv(csv)
        ax.plot(df["Q"], df["F_kT"], lw=1.2, alpha=0.75, label=protein_dir.name)
    ax.set_title(group)
    ax.set_xlabel("Reaction Coordinate (Q)")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("Free Energy (kT)")
axes[-1].legend(fontsize=6, ncol=2, loc="upper center", frameon=False)
fig.tight_layout()

summary_dir = SIM_ROOT / "logs"
fig.savefig(summary_dir / "QG_free_energy_by_group.png", dpi=300)
print("wrote", summary_dir / "QG_free_energy_by_group.png")
if SHOW_PLOTS:
    plt.show()
plt.close(fig)

wrote /storage/group/epo2/default/qzv5006/Research_data/Aging_project/R1_analysis/minor5/sim/logs/QG_free_energy_by_group.png


In [10]:

# Combined manifest for the record.
summary = manifest.merge(fe, on=["group", "protein", "n_replicas"], how="left")
summary.to_csv(SIM_ROOT / "logs" / "QG_manifest.csv", index=False)
print("wrote", SIM_ROOT / "logs" / "QG_manifest.csv")
summary

wrote /storage/group/epo2/default/qzv5006/Research_data/Aging_project/R1_analysis/minor5/sim/logs/QG_manifest.csv


,group,protein,n_replicas,frames,status,note,Q_mean,Q_tail_mean,Q_at_Fmin,frac_traj_tailQ_gt_0p6,G_max
0,NSC_ENT,01_P38910,35,19999,exists (skipped),,0.896107,0.909025,0.940000,1.00,0.07853
1,NSC_ENT,02_P38828,50,19999,exists (skipped),,0.812044,0.821468,0.806667,1.00,0.11009
2,NSC_ENT,03_P38146,50,19999,exists (skipped),,0.858756,0.901779,0.900000,1.00,0.27487
3,NSC_ENT,04_P32835,50,19999,exists (skipped),,0.848285,0.935700,0.953333,1.00,0.26360
4,NSC_ENT,05_P38858,50,19999,exists (skipped),,0.807711,0.954413,0.953333,1.00,0.19904
5,NSC_ENT,06_P03962,50,19999,exists (skipped),,0.913866,0.948938,0.966667,1.00,0.09926
6,NSC_ENT,07_P38816,50,19999,exists (skipped),,0.852297,0.902342,0.913333,1.00,0.13974
7,NSC_ENT,08_P32179,50,19999,exists (skipped),,0.861748,0.900880,0.980000,1.00,0.13900
8,NSC_ENT,09_P28834,50,19999,exists (skipped),,0.863338,0.911951,0.953333,1.00,0.21123
9,NSC_ENT,10_Q12283,50,19999,exists (skipped),,0.870860,0.936888,0.953333,1.00,0.13254


## Outputs| path | content ||---|---|| `<group>/<protein>/analysis/QG.npy` | `(n, 19999, 2)` float64 — `[:, :, 0] = Q`, `[:, :, 1] = G` || `<group>/<protein>/analysis/QG_info.json` | replica indices per row, shape, source path || `<group>/<protein>/analysis/Q_free_energy.csv` / `.png` | `F(Q)` in kT on the fixed `Q_RANGE` grid || `<group>/<protein>/analysis/G_free_energy.csv` / `.png` | `F(G)` in kT, data-driven range || `logs/QG_free_energy_by_group.png` | all `F(Q)` curves, one panel per group || `logs/QG_manifest.csv` | per-protein replica counts, shapes, folding diagnostics |Set `FORCE = True` in the config cell to rebuild arrays that already exist.